<a href="https://colab.research.google.com/github/harshchaudhary11/Machine-Learning/blob/main/RAG%20works%20by%20combining%20vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu
!pip install sentence-transformers
!pip install transformers
!pip install langchain==0.1.16

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

In [2]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

documents = [
    "RAG combines retrieval and generation.",
    "It reduces hallucinations using external data.",
    "FAISS enables fast similarity search.",
    "Embeddings represent semantic meaning.",
    "RAG improves LLM accuracy."
]

doc_embeddings = embed_model.encode(documents)

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings).astype('float32'))

print(f"Indexed {index.ntotal} documents.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 5 documents.


In [3]:
def semantic_search(query_embedding, top_k=3):
    distances, indices = index.search(query_embedding, top_k)
    return indices

In [4]:
query_embedding = embed_model.encode(["What is RAG?"]).astype('float32')
retrieved_indices = semantic_search(query_embedding)

print(retrieved_indices)

[[0 4 1]]


In [5]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
prompt_template = PromptTemplate(
    input_variables=["question", "context"],
    template="Question: {question}\nContext: {context}\nAnswer:"
)

In [7]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=False
)

def chat(question):

    query_embedding = embed_model.encode([question]).astype("float32")
    retrieved_indices = semantic_search(query_embedding)

    context_texts = [documents[i] for i in retrieved_indices[0]]
    context = "\n".join(context_texts)

    chat_history = memory.load_memory_variables({}).get("chat_history", "")

    prompt = prompt_template.format(
        question=question,
        context=context
    )

    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs,
        max_new_tokens=80,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    memory.chat_memory.add_user_message(question)
    memory.chat_memory.add_ai_message(response)

    return response

In [9]:
print(chat(input("What is RAG?")))
print(chat("How does retrieval help LLMs?"))

What is RAG?what is RAG
Question: what is RAG
Context: RAG combines retrieval and generation.
RAG improves LLM accuracy.
Embeddings represent semantic meaning.
Answer: RAG is a semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic semantic
Question: How does retrieval help LLMs?
Context: RAG improves LLM accuracy.
RAG combines retrieval and generation.
It reduces hallucination